# Provenance concept test (Ollama)

**Idea:** every generated item must point to the source rule that justifies it.
We test three things on a tiny BRD:
1. Does the item have a link to a rule that exists? (simple check, no AI)
2. Does that rule really justify the item? (AI judge)
3. Which faults slip through?

Before you start, in a terminal: `ollama pull llama3.1:8b` (Ollama app must be running).

In [ ]:
%pip install -q ollama

In [1]:
import json, ollama

MODEL = "llama3.1:8b"   # any model you pulled with `ollama pull`

def ask_json(prompt):
    """Send a prompt to the local model and parse its JSON answer."""
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"temperature": 0},
    )
    return json.loads(reply["message"]["content"])

## 1. A tiny BRD with rule IDs

In [2]:
BRD = {
    "R1": "Refunds over 500 EGP require manager approval.",
    "R2": "Refunds under 500 EGP are approved by the cashier.",
    "R3": "All refunds must be logged in the finance system within 24 hours.",
}
rules_text = "\n".join(f"{k}: {v}" for k, v in BRD.items())

## 2. Ask the model to generate workflow items, each with a source rule

In [4]:
prompt = f"""From the business rules below, list the workflow states and user roles needed.
Return JSON like {{"items": [{{"name": "...", "type": "state or role", "source": "R1"}}]}}.
Every item must name the rule id it comes from. Use only rule ids that exist.

Rules:
{rules_text}"""

items = ask_json(prompt)["items"]
for it in items:
    print(it)

ResponseError: model 'llama3.1:8b' not found (status code: 404)

## 3. Add known faults (so we know what the checks should catch)

In [ ]:
faults = [
    {"name": "Auditor",           "type": "role",  "source": None},   # invented, no source
    {"name": "Director Approval", "type": "state", "source": "R9"},   # source id does not exist
    {"name": "Cashier Approval",  "type": "state", "source": "R1"},   # wrong source (R1 is about managers)
]
all_items = items + faults

## 4. Check 1: does the link exist? (no AI)

In [ ]:
def link_exists(item):
    src = item.get("source")
    return isinstance(src, str) and src in BRD

for it in all_items:
    print("OK     " if link_exists(it) else "NO LINK", "|", it["name"], "->", it.get("source"))

## 5. Check 2: does the source really justify the item? (AI judge)

In [ ]:
def judge(item):
    p = f"""Rule: {BRD[item['source']]}
Workflow item: {item['name']} ({item['type']})
Does this rule justify that this item is needed?
Return JSON: {{"supported": true or false, "reason": "one short sentence"}}"""
    return ask_json(p)

for it in all_items:
    if link_exists(it):
        r = judge(it)
        label = "SUPPORTED    " if r.get("supported") else "NOT SUPPORTED"
        print(label, "|", it["name"], "->", it["source"], "|", r.get("reason"))

Agent_Decision_2 is wrong
  └─ check Response_2 against Prompt_2 + Scores_2 + Control_Result_2
       ├─ the response contradicts its recorded inputs  → hallucination (the LLM is at fault)
       └─ the response matches its inputs → keep tracing upstream
            └─ Model_Evaluation_2 → Physics_Model_2 → Sensor_Data_2 → find the real fault

## 6. Questions to answer
- Which faults did check 1 catch? Which needed the judge?
- Did the judge catch `Cashier Approval -> R1`? (R1 talks about managers, the cashier comes from R2.)
- Run the judge cell 3 times. Is it consistent? Try a smaller model (`llama3.2:3b`). Does it get worse?
- Which items did the model generate on its own that look wrong or invented?